# Syllable Counting

Syllable counts can be a useful feature particularly in the analysis of verse corpora. This notebook is intended as a starting-point for this task as it pertains to low-resource languages like Old English.

As long as we are interested merely in the number of syllables per metrical unit rather than exact boundaries, the task of counting syllables may almost be as straightforward as counting vowels. However, two major problems remain:

1. In languages/documents that mix &lt;u, v&gt; and/or &lt;i, j&gt; or use one of each symbol for both vowel and consonant, we may need to judge by context;
2. The existence of diphthongs means syllable boundaries still matter when distinguishing between diphthongs and adjacent syllables.

Accurate syllabification will accordingly differ between languages. But let's at least take a few steps towards an Old English syllable counter.

In [1]:
document = 'hwæt we gardena in geardagum ðeodcyninga ðrym gefrunon hu þa æþelingas ' \
'ellen fremedon oft scyld scefing sceaðena ðreatum monegum mægðum meodosetla ofteah egsode ' \
'eorlas syððan ærest wearð feasceaft funden'
tokens = document.split()

In [2]:
vowels = 'aæeiouy'
diphthongs = ['au', 'æa', 'ea', 'eo', 'io', 'iu']
second_elements = list(set([i[1] for i in diphthongs]))

In [3]:
second_elements

['o', 'a', 'u']

In Old English, &lt;u&gt; or &lt;v&gt; nearly always represents a vowel: [v] was represented by &lt;f&gt;, its realization clear from its voiced context, and at any rate not phonemic. Likewise &lt;i&gt; should present no difficulties: where it precedes another vowel it is normally part of a diphthong, so as long as we account for diphthongs we can hardly miscount it.

Instead, the main difficulty will be in distinguishing between diphthongs and adjacent syllables. For greatest accuracy, we would have to compile a dictionary of word stems placing the vowels of adjacent syllables side by side, such as the regex pattern `"^gea[hx]"`, which matches forms of the verb _axian_ containing the *ge*-prefix. In the absence of such a dictionary, we can either undercount (by considering all _ea_-sequences diphthongs) or overcount (by removing this sequence from our diphthong dictionary), whichever gets us closer to an accurate count.

In [4]:
def syllable_count(form):
    # We initialize a vowel counter:
    counter = 0
    # And a character position counter:
    position = 0
    # Now we cycle through the token's characters:
    for character in form:
        if character in vowels:
            # If the current vowel combined with the previous forms a known diphthong, 
            # count them as one vowel:
            if character in second_elements and position > 0 and form[position-1] \
                + character in diphthongs:
                position += 1
                continue
            # Otherwise count the current vowel separately:
            else:
                counter += 1
                position += 1
        else:
            # If we're not on a vowel, just advance the position:
            position += 1
    return counter

In [5]:
for token in tokens:
    result = syllable_count(token)
    print(f"{token}: {result}")

hwæt: 1
we: 1
gardena: 3
in: 1
geardagum: 3
ðeodcyninga: 4
ðrym: 1
gefrunon: 3
hu: 1
þa: 1
æþelingas: 4
ellen: 2
fremedon: 3
oft: 1
scyld: 1
scefing: 2
sceaðena: 3
ðreatum: 2
monegum: 3
mægðum: 2
meodosetla: 4
ofteah: 2
egsode: 3
eorlas: 2
syððan: 2
ærest: 2
wearð: 1
feasceaft: 2
funden: 2


That looks decent at first sight. But of course it fails in cases like the following:

In [6]:
problems = ['geahsodon', 'geopenað']
for token in problems:
    result = syllable_count(token)
    print(f"{token}: {result}")

geahsodon: 3
geopenað: 3


The difficulty is that the vast majority of forms in _beo-_, _gea-_ and _geo-_ nevertheless have diphthongs: consider _geard_, _gearwe_, _geond_, _geogoþ_, and _georne_, to say nothing of _beon_. So it would be vastly more accurate to ignore these and undercount syllables than to count two syllables for each initial sequence _gea-_, _geo-_, etc. The only realistic alternative is to draw on an extensive lexicon of forms explicitly marked as consisting of multiple morphemes. A middle way might be to draw on e.g. [Nerthus](http://www.nerthusproject.com) and assume any past participles starting in _ge-_ plus vowel require us to count an extra syllable, or draw on e.g. YCOE and assume any verb starting in _ge-_ or _be-_ plus vowel (unless it be `"beo(n|ð|þ|$)"`) has an extra syllable.

Incidentally, if you're running a version of Python between 3.7 and 3.9, you should be able to install CLTK version 1.5.0 (`pip install cltk==1.5.0`), which has an Old English syllabifier built in. Let's see how it fares:

In [7]:
from cltk.phonology.ang.phonology import OldEnglishSyllabifier
syll = OldEnglishSyllabifier()

In [8]:
for token in tokens:
    print(syll(token))

['hwæt']
['we']
['gar', 'den', 'a']
['in']
['gear', 'da', 'gum']
['ðeodcy', 'ninga']
['ðrym']
['gef', 'ru', 'non']
['hu']
['þa']
['æ', 'þe', 'lingas']
['ellen']
['fre', 'me', 'don']
['oft']
['scyld']
['sce', 'fing']
['scea', 'ðen', 'a']
['ðrea', 'tum']
['mo', 'ne', 'gum']
['mægðum']
['meo', 'do', 'setl', 'a']
['of', 'teah']
['eg', 'sod', 'e']
['eor', 'las']
['syððan']
['æ', 'rest']
['wearð']
['feas', 'ceaft']
['funden']


In [9]:
for token in problems:
    print(syll(token))

['geahso', 'don']
['geo', 'pe', 'nað']


That's interesting: CLTK's syllabifier actually does worse at syllable counting than our few lines of code, while doing no better on prefixes. We can dig into CLTK's code (look for `lib/python3.X/site-packages/cltk/phonology/syllabify.py` in your Python folder, and also glance at `syllabify.py` as contained in the folder `ang/` one level down), where we find that the syllabifier consists of 671 lines of code (!) plus language-specific sonority rankings; the script claims to implement the maximum onset principle as well as the sonority sequence principle, but it is clear from e.g. its treatment of "gardena" and "sceaðena" above that it occasionally fails at the first of these principles; its analysis of "ðeodcyninga" suggests it treats &lt;y&gt; as a consonant! Implementing a full new syllabifier goes beyond the aims of the present exercise, but one imagines a better syllabifier is feasible.

NB there is another set of complications involved in Old Germanic metrical syllabification. On the one hand, __resolution__ treats some pairs of short syllables as just one metrical syllable (affected in the above sample, for instance, are "**freme**don," "**æðe**lingas," "**sceaðe**na," "**mone**gum," and "**meodo**setla," but in other instances, such as "gar**dena**," resolution is blocked by other rules that take precedence!); on the other, some cases of __contraction__ e.g. in verbs like _seon_ (from _\*sehan_) need to be read as uncontracted. These questions can only be solved with elaborate algorithms involving word-lists and automated metrical analysis, or at the very least a large database of verses labelled for Sievers type (or, better, Bliss). Perhaps something for a future project! For present purposes, if you involve syllable counting in your project, please also include a disclaimer saying that you have not considered such advanced dynamics as resolution or suspension of contraction.

## Implementation

Since the assignment was to count the average number of syllables to a metrical unit, let's now gather our text into halflines and rewrite our function to accept spaced sequences of words (halflines or full lines, whichever you choose to feed it) and average out the counts per unit:

In [10]:
verses = [
    'hwæt we gardena',
    'in geardagum',
    'ðeodcyninga',
    'ðrym gefrunon',
    'hu þa æþelingas',
    'ellen fremedon',
    'oft scyld scefing,'
    'sceaðena ðreatum',
    'monegum mægðum',
    'meodosetla ofteah',
    'egsode eorlas',
    'syððan ærest wearð',
    'feasceaft funden'
    ]

def avg_syllables(doc):
    counts = []
    for verse in doc:
        # We'll keep a vowel counter per halfline, to which we add the tally for each token:
        syllables = 0
        tokens = verse.split()
        for form in tokens:
            # We initialize a vowel counter:
            counter = 0
            # And a character position counter:
            position = 0
            # Now we cycle through the token's characters:
            for character in form:
                if character in vowels:
                    # If the current vowel combined with the previous forms a known diphthong, 
                    # count them as one vowel:
                    if character in second_elements and position > 0 and form[position-1] \
                        + character in diphthongs:
                        position += 1
                        continue
                    # Otherwise count the current vowel separately:
                    else:
                        counter += 1
                        position += 1
                else:
                    # If we're not on a vowel, just advance the position:
                    position += 1
            syllables += counter
        counts.append(syllables)
    # Finally, sum the counts and divide by the length of the list of counts:
    return round(sum(counts)/len(counts), 2)

In [11]:
avg_syllables(verses)

5.17